# ðŸ¦´ Bone Fracture Detection â€” YOLOv8m Training

**Dataset:** BoneFractureYolo8 (7-class detection, polygon labels)  
**Model:** YOLOv8m (pre-trained on COCO)  
**GPU:** 2Ã— NVIDIA Tesla T4 (Kaggle)  

This notebook runs on the **Kaggle remote kernel**. All training happens on Kaggle GPUs.

## Phase 0 â€” Environment Verification

We always verify GPU availability before doing anything. If CUDA is not available, we stop.

In [ ]:
import subprocess, torch, os

# ---- nvidia-smi ----
print('=' * 60)
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else 'nvidia-smi not found')

# ---- PyTorch CUDA check ----
print('=' * 60)
print(f'PyTorch version  : {torch.__version__}')
print(f'CUDA available   : {torch.cuda.is_available()}')

if not torch.cuda.is_available():
    raise RuntimeError(
        'CUDA is NOT available. Do not train on CPU. '
        'Ensure this notebook is running on the Kaggle GPU kernel.'
    )

num_gpus = torch.cuda.device_count()
print(f'Number of GPUs   : {num_gpus}')
for i in range(num_gpus):
    props = torch.cuda.get_device_properties(i)
    free_mem, total_mem = torch.cuda.mem_get_info(i)
    print(f'  GPU {i}: {props.name}')
    print(f'         Total VRAM : {total_mem / 1e9:.1f} GB')
    print(f'         Free  VRAM : {free_mem  / 1e9:.1f} GB')

DEVICE = '0,1' if num_gpus >= 2 else '0'
print(f'\nTraining device  : cuda:{DEVICE}')

## Phase 1 â€” Install Dependencies & Setup Paths

In [ ]:
# Install ultralytics (Kaggle may already have it, but pin version)
import subprocess
subprocess.run(['pip', 'install', '-q', 'ultralytics>=8.2.0'], check=True)
print('Ultralytics installed.')

In [ ]:
import os, shutil, yaml
from pathlib import Path

# ── Kaggle paths ──────────────────────────────────────────────────────────────
KAGGLE_INPUT   = Path('/kaggle/input')
WORKING_DIR    = Path('/kaggle/working')
CHECKPOINT_DIR = WORKING_DIR / 'checkpoints'
OUTPUT_DIR     = WORKING_DIR / 'outputs'

for d in [CHECKPOINT_DIR, OUTPUT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ── Find the pre-extracted BoneFractureYolo8 folder ──────────────────────────
# On Kaggle the dataset is already extracted — no zip needed.
# We search /kaggle/input recursively for the BoneFractureYolo8 folder.
def find_dataset_dir(name='BoneFractureYolo8'):
    # Check direct Kaggle input subdirectories first
    for candidate in KAGGLE_INPUT.rglob(name):
        if candidate.is_dir() and (candidate / 'train').exists():
            return candidate
    # Fallback for local development
    local = Path('data') / name
    if local.exists():
        return local
    raise FileNotFoundError(
        f'Could not find dataset folder "{name}" under {KAGGLE_INPUT}.\n'
        f'Contents of /kaggle/input:\n' +
        '\n'.join(str(p) for p in sorted(KAGGLE_INPUT.rglob('*'))[:40])
    )

EXTRACT_DIR = find_dataset_dir('BoneFractureYolo8')
print(f'Dataset found at: {EXTRACT_DIR}')

# Verify folder structure
for split in ['train', 'valid', 'test']:
    img_dir = EXTRACT_DIR / split / 'images'
    lbl_dir = EXTRACT_DIR / split / 'labels'
    img_count = len(list(img_dir.glob('*.jpg'))) if img_dir.exists() else 0
    lbl_count = len(list(lbl_dir.glob('*.txt'))) if lbl_dir.exists() else 0
    print(f'  {split:5s}: {img_count} images, {lbl_count} labels')


## Phase 2 â€” Extract & Prepare Dataset

In [ ]:
# Dataset is already extracted on Kaggle — no unzipping needed.
# This cell confirms the structure is correct before writing data.yaml.

print(f'Using dataset at: {EXTRACT_DIR}')
print()

# Sanity check — make sure images and labels are paired
for split in ['train', 'valid', 'test']:
    img_dir = EXTRACT_DIR / split / 'images'
    lbl_dir = EXTRACT_DIR / split / 'labels'
    imgs = set(p.stem for p in img_dir.glob('*.jpg'))
    lbls = set(p.stem for p in lbl_dir.glob('*.txt'))
    paired   = len(imgs & lbls)
    img_only = len(imgs - lbls)
    lbl_only = len(lbls - imgs)
    print(f'{split:5s}: {len(imgs)} images | {len(lbls)} labels | '
          f'{paired} paired | {img_only} imgs-without-label | {lbl_only} labels-without-img')

print('\nDataset structure verified.')


In [ ]:
# Write a corrected data.yaml with absolute paths.
# Ultralytics needs this to resolve train/val/test image directories.

DATA_YAML = WORKING_DIR / 'data.yaml'

CLASS_NAMES = [
    'elbow positive',
    'fingers positive',
    'forearm fracture',
    'humerus fracture',
    'humerus',
    'shoulder fracture',
    'wrist positive',
]

data_config = {
    'path'  : str(EXTRACT_DIR),   # root: .../BoneFractureYolo8
    'train' : 'train/images',
    'val'   : 'valid/images',
    'test'  : 'test/images',
    'nc'    : 7,
    'names' : CLASS_NAMES,
}

with open(DATA_YAML, 'w') as f:
    yaml.dump(data_config, f, default_flow_style=False, sort_keys=False)

print('data.yaml written to:', DATA_YAML)
print()
print(DATA_YAML.read_text())


## Phase 3 â€” Dataset Analysis

Before training, we check class distribution and a few sample images to understand what the model will see.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
from collections import Counter
from PIL import Image

CLASS_NAMES = data_config['names']
COLORS = plt.cm.Set1(np.linspace(0, 1, len(CLASS_NAMES)))

# Count class distribution across all splits
split_counts = {}
for split in ['train', 'valid', 'test']:
    lbl_dir = EXTRACT_DIR / split / 'labels'
    counter = Counter()
    for lbl_file in lbl_dir.glob('*.txt'):
        for line in lbl_file.read_text().strip().splitlines():
            if line.strip():
                counter[int(line.split()[0])] += 1
    split_counts[split] = counter

# Plot class distribution
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (split, counter) in zip(axes, split_counts.items()):
    counts = [counter.get(i, 0) for i in range(len(CLASS_NAMES))]
    bars = ax.bar(CLASS_NAMES, counts, color=COLORS)
    ax.set_title(f'{split.capitalize()} Set Class Distribution', fontsize=12)
    ax.set_xlabel('Class')
    ax.set_ylabel('Annotation Count')
    ax.tick_params(axis='x', rotation=30)
    for bar, count in zip(bars, counts):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                str(count), ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'class_distribution.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved: class_distribution.png')

In [ ]:
# Visualize a few training samples with their bounding boxes
import random

random.seed(42)
img_dir = EXTRACT_DIR / 'train' / 'images'
lbl_dir = EXTRACT_DIR / 'train' / 'labels'

sample_imgs = random.sample(list(img_dir.glob('*.jpg')), k=6)

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for ax, img_path in zip(axes, sample_imgs):
    img = Image.open(img_path).convert('RGB')
    W, H = img.size
    ax.imshow(img)
    ax.set_title(img_path.name[:30], fontsize=7)
    ax.axis('off')

    lbl_path = lbl_dir / (img_path.stem + '.txt')
    if lbl_path.exists():
        for line in lbl_path.read_text().strip().splitlines():
            parts = list(map(float, line.split()))
            cls_id = int(parts[0])
            coords = parts[1:]  # polygon or bbox normalized

            if len(coords) == 4:
                # Standard YOLO bbox: cx cy w h (normalized)
                cx, cy, w, h = coords
                x1, y1 = (cx - w/2) * W, (cy - h/2) * H
                rect = patches.Rectangle(
                    (x1, y1), w * W, h * H,
                    linewidth=2, edgecolor=COLORS[cls_id], facecolor='none'
                )
                ax.add_patch(rect)
                label_x, label_y = x1, y1 - 5
            else:
                # Polygon: pairs of (x, y) normalized
                xs = [coords[i]     * W for i in range(0, len(coords), 2)]
                ys = [coords[i + 1] * H for i in range(0, len(coords), 2)]
                poly = plt.Polygon(
                    list(zip(xs, ys)),
                    fill=False, edgecolor=COLORS[cls_id], linewidth=2
                )
                ax.add_patch(poly)
                label_x, label_y = min(xs), min(ys) - 5

            ax.text(
                label_x, label_y,
                CLASS_NAMES[cls_id], color=COLORS[cls_id],
                fontsize=7, fontweight='bold',
                bbox=dict(facecolor='white', alpha=0.5, pad=1)
            )

plt.suptitle('Sample Training Images with Annotations', fontsize=14)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'sample_images.png', dpi=100, bbox_inches='tight')
plt.show()
print('Saved: sample_images.png')

## Phase 4 â€” Small Smoke Test (2 Epochs, 5% Data)

Before full training, run 2 epochs on a small subset to catch any errors early.

> If this cell succeeds without errors, GPU is working and the pipeline is correct.

In [ ]:
import random, shutil, yaml
from pathlib import Path
from ultralytics import YOLO

# Create a small 5% subset for the smoke test
SMOKE_DIR = WORKING_DIR / 'smoke_test_data'

if not SMOKE_DIR.exists():
    random.seed(42)
    for split in ['train', 'valid']:
        img_src = EXTRACT_DIR / split / 'images'
        lbl_src = EXTRACT_DIR / split / 'labels'
        img_dst = SMOKE_DIR / split / 'images'
        lbl_dst = SMOKE_DIR / split / 'labels'
        img_dst.mkdir(parents=True, exist_ok=True)
        lbl_dst.mkdir(parents=True, exist_ok=True)

        all_imgs = list(img_src.glob('*.jpg'))
        k = max(30, int(len(all_imgs) * 0.05))  # 5% but at least 30
        subset = random.sample(all_imgs, k=min(k, len(all_imgs)))

        for img in subset:
            shutil.copy(img, img_dst / img.name)
            lbl = lbl_src / (img.stem + '.txt')
            if lbl.exists():
                shutil.copy(lbl, lbl_dst / lbl.name)

        print(f'  Smoke {split}: {len(subset)} images copied')
else:
    print('Smoke test data already exists — skipping copy.')

# Write smoke test yaml
SMOKE_YAML = WORKING_DIR / 'smoke_data.yaml'
smoke_config = {
    'path'  : str(SMOKE_DIR),
    'train' : 'train/images',
    'val'   : 'valid/images',
    'nc'    : 7,
    'names' : CLASS_NAMES,
}
with open(SMOKE_YAML, 'w') as f:
    yaml.dump(smoke_config, f, default_flow_style=False, sort_keys=False)

print(f'\nRunning smoke test (2 epochs, ~5% data) on device: {DEVICE} ...')

# YOLOv8m — downloads pretrained weights automatically (needs internet ON in Kaggle)
smoke_model = YOLO('yolov8m.pt')

smoke_results = smoke_model.train(
    data       = str(SMOKE_YAML),
    epochs     = 2,
    imgsz      = 640,
    batch      = 8,
    device     = DEVICE,
    project    = str(WORKING_DIR / 'runs'),
    name       = 'smoke_test',
    exist_ok   = True,
    save       = False,
    verbose    = True,
    amp        = True,
    seed       = 42,
    plots      = False,
)

print('\n=== Smoke Test PASSED ===')
print(f'GPU used     : {DEVICE}')
print('Next step    : Run Phase 5 (full training)')


## Phase 5 â€” Full Training Run

Now we train YOLOv8m on the full dataset for 50 epochs.

**Key settings:**
- Image size: 640
- Batch: 16 (auto-reduces if OOM)
- AMP: enabled (faster + less VRAM)
- Early stopping: patience=10
- Best model auto-saved by Ultralytics

In [ ]:
from ultralytics import YOLO
import torch, yaml
from pathlib import Path

# Reproducibility
SEED = 42
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# ── Hyperparameters ───────────────────────────────────────────────────────────
EPOCHS   = 50
IMGSZ    = 640
BATCH    = 16   # reduce to 8 if you get CUDA OOM
LR0      = 0.01
LRF      = 0.01
PATIENCE = 10
RUN_NAME = 'fracture_yolov8m'

hparams = dict(
    model='yolov8m', epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH,
    lr0=LR0, lrf=LRF, patience=PATIENCE, device=DEVICE, amp=True, seed=SEED
)
with open(OUTPUT_DIR / 'hparams.yaml', 'w') as f:
    yaml.dump(hparams, f)
print('Hyperparameters saved.')

# Load YOLOv8m with COCO pre-trained weights
model = YOLO('yolov8m.pt')

print(f'Starting full training on device: {DEVICE}')
print(f'Dataset yaml : {DATA_YAML}')
print(f'Epochs       : {EPOCHS}  |  Batch: {BATCH}  |  ImgSz: {IMGSZ}')
print('=' * 60)

results = model.train(
    data       = str(DATA_YAML),
    epochs     = EPOCHS,
    imgsz      = IMGSZ,
    batch      = BATCH,
    device     = DEVICE,
    project    = str(WORKING_DIR / 'runs'),
    name       = RUN_NAME,
    exist_ok   = True,
    save       = True,
    save_period= 5,
    patience   = PATIENCE,
    amp        = True,
    lr0        = LR0,
    lrf        = LRF,
    cos_lr     = True,
    seed       = SEED,
    plots      = True,
    verbose    = True,
    mosaic     = 1.0,
    flipud     = 0.0,
    fliplr     = 0.5,
    hsv_h      = 0.015,
    hsv_s      = 0.7,
    hsv_v      = 0.4,
)

BEST_MODEL = Path(results.save_dir) / 'weights' / 'best.pt'
print('\n=== Training Complete ===')
print(f'Best model   : {BEST_MODEL}')
print(f'Run dir      : {results.save_dir}')


## Phase 6 â€” Evaluation on Test Set

We load the **best checkpoint** (not the last epoch) and evaluate on the held-out test set.

In [ ]:
from ultralytics import YOLO
from pathlib import Path

# Load the best model saved during training
BEST_MODEL = WORKING_DIR / 'runs' / 'fracture_yolov8m' / 'weights' / 'best.pt'
if not BEST_MODEL.exists():
    raise FileNotFoundError(f'best.pt not found at {BEST_MODEL}. Run Phase 5 first.')

best_model = YOLO(str(BEST_MODEL))

print('Evaluating best model on TEST set ...')
test_metrics = best_model.val(
    data    = str(DATA_YAML),
    split   = 'test',
    device  = DEVICE,
    imgsz   = IMGSZ,
    batch   = BATCH,
    verbose = True,
    plots   = True,
    save_json = True,
    project = str(OUTPUT_DIR),
    name    = 'test_eval',
)

print('\n=== TEST SET RESULTS ===')
print(f'mAP@50       : {test_metrics.box.map50:.4f}')
print(f'mAP@50-95    : {test_metrics.box.map:.4f}')
print(f'Precision    : {test_metrics.box.mp:.4f}')
print(f'Recall       : {test_metrics.box.mr:.4f}')
print()
print('Per-class AP@50:')
for i, (name, ap) in enumerate(zip(CLASS_NAMES, test_metrics.box.ap50)):
    print(f'  {name:25s}: {ap:.4f}')

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from pathlib import Path

RUN_DIR = WORKING_DIR / 'runs' / 'fracture_yolov8m'

# â”€â”€ Training curves â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
results_png = RUN_DIR / 'results.png'
if results_png.exists():
    img = mpimg.imread(str(results_png))
    plt.figure(figsize=(16, 10))
    plt.imshow(img)
    plt.axis('off')
    plt.title('Training & Validation Curves', fontsize=14)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'training_curves.png', dpi=120, bbox_inches='tight')
    plt.show()
else:
    print(f'results.png not found at {results_png}')

# â”€â”€ Confusion matrix â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
cm_png = RUN_DIR / 'confusion_matrix_normalized.png'
if cm_png.exists():
    img = mpimg.imread(str(cm_png))
    plt.figure(figsize=(10, 8))
    plt.imshow(img)
    plt.axis('off')
    plt.title('Confusion Matrix (Normalized)', fontsize=14)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'confusion_matrix.png', dpi=120, bbox_inches='tight')
    plt.show()

# â”€â”€ PR curve â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
pr_png = RUN_DIR / 'PR_curve.png'
if pr_png.exists():
    img = mpimg.imread(str(pr_png))
    plt.figure(figsize=(10, 7))
    plt.imshow(img)
    plt.axis('off')
    plt.title('Precision-Recall Curve', fontsize=14)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'pr_curve.png', dpi=120, bbox_inches='tight')
    plt.show()

In [ ]:
import random
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np

# Run inference on 6 test images and visualize predictions
test_img_dir = EXTRACT_DIR / 'test' / 'images'
test_imgs = random.sample(list(test_img_dir.glob('*.jpg')), k=6)

preds = best_model.predict(
    source    = [str(p) for p in test_imgs],
    imgsz     = IMGSZ,
    device    = DEVICE,
    conf      = 0.25,
    iou       = 0.45,
    save      = False,
    verbose   = False,
)

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for ax, result in zip(axes, preds):
    # result.plot() returns a BGR numpy array
    img_bgr = result.plot()
    img_rgb = img_bgr[..., ::-1]  # BGR -> RGB
    ax.imshow(img_rgb)
    ax.axis('off')
    n_det = len(result.boxes) if result.boxes is not None else 0
    ax.set_title(f'{Path(result.path).name[:25]} | {n_det} detection(s)', fontsize=8)

plt.suptitle('Model Predictions on Test Images (confâ‰¥0.25)', fontsize=13)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'test_predictions.png', dpi=100, bbox_inches='tight')
plt.show()
print('Saved: test_predictions.png')

## Phase 7 â€” Results Summary & Next Steps

In [ ]:
print('=' * 60)
print('TRAINING SUMMARY')
print('=' * 60)
print(f'Model          : YOLOv8m')
print(f'Dataset        : BoneFractureYolo8 (7 classes)')
print(f'Dataset path   : {EXTRACT_DIR}')
print(f'Train images   : 3,631')
print(f'Val   images   : 348')
print(f'Test  images   : 169')
print()
print('TEST SET METRICS:')
print(f'  mAP@50       : {test_metrics.box.map50:.4f}')
print(f'  mAP@50-95    : {test_metrics.box.map:.4f}')
print(f'  Precision    : {test_metrics.box.mp:.4f}')
print(f'  Recall       : {test_metrics.box.mr:.4f}')
print()
print('Per-class AP@50:')
for name, ap in zip(CLASS_NAMES, test_metrics.box.ap50):
    print(f'  {name:25s}: {ap:.4f}')
print()
print('FILES TO DOWNLOAD from /kaggle/working/ :')
print(f'  best.pt          -> {BEST_MODEL}')
print(f'  hparams.yaml     -> {OUTPUT_DIR}/hparams.yaml')
print(f'  class_dist.png   -> {OUTPUT_DIR}/class_distribution.png')
print(f'  train_curves.png -> {OUTPUT_DIR}/training_curves.png')
print(f'  conf_matrix.png  -> {OUTPUT_DIR}/confusion_matrix.png')
print(f'  pr_curve.png     -> {OUTPUT_DIR}/pr_curve.png')
print(f'  predictions.png  -> {OUTPUT_DIR}/test_predictions.png')


## Potential Improvements (Run After Seeing Results)

Based on known dataset properties (check actual results to confirm):

| Issue | Fix |
|-------|-----|
| Class 3 (humerus fracture) has only 3 samples | Collect more data or exclude from evaluation |
| Class imbalance across all classes | Try `cls` loss weight tuning |
| Low mAP overall | Try YOLOv8l or YOLOv8x (larger model) |
| Overfitting (val loss rises) | Add dropout, reduce model size, more augmentation |
| Underfitting (both losses plateau high) | Train longer, larger model, lower LR |
| Slow convergence | Try SGD â†’ AdamW optimizer |

After reviewing the confusion matrix:
- Classes being confused with each other â†’ check if label quality is good
- Many false negatives â†’ lower confidence threshold
- Many false positives â†’ raise confidence threshold or train longer